# 02 — Rectify a detector image for Dioptas / pyFAI / GSAS-II

★ **This is the package's headline interop feature.**

`correct_image()` bakes the MIDAS geometry (tilts + 15-parameter
distortion + panel shifts) into pixel values, producing a TIFF that
downstream tools can integrate with simple flat-detector math:

    R = sqrt((y - ybc)² + (z - zbc)²) · pixel_size

No MIDAS-specific calibration round-trip required on the other side.

**Install**: `pip install midas-integrate` (pulls midas-auto-calibrate too).

In [1]:
import shutil, tempfile
from pathlib import Path

import numpy as np

import midas_auto_calibrate as mac
import midas_integrate as mi

## Step 1 — Calibrate (or load a prior calibration)

`correct_image` takes either a `midas_auto_calibrate.DetectorGeometry`
or a `midas_integrate.IntegrationConfig`. Either way, you need the
refined tilts + distortion parameters — from `mac.auto_calibrate()`
or a saved JSON.

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mi_correct_'))
cal_dir = workdir / 'cal'; cal_dir.mkdir()
img_path = cal_dir / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img_path)
shutil.copy(mac.data.CEO2_PILATUS_DARK, cal_dir / 'dark.tif')
shutil.copy(mac.data.CEO2_PILATUS_MASK, cal_dir / 'mask_upd.tif')

result = mac.auto_calibrate(
    mac.CalibrationConfig(
        material='CeO2',
        lattice_params=(5.4116,) * 3 + (90.0,) * 3,
        wavelength=0.172973, pixel_size=172.0,
        lsd=657_436.9, ybc=685.5, zbc=921.0,
        nr_pixels_y=1475, nr_pixels_z=1679,
        dark_file='dark.tif', mask_file='mask_upd.tif',
        im_trans_opt=[2], n_iterations=5,
    ),
    img_path, work_dir=cal_dir, n_cpus=4,
)
print(f'tilts  : ty={result.geometry.ty:+.5f}°, tz={result.geometry.tz:+.5f}°')
print(f'p0..p4 : {result.geometry.distortion[:5]}')

tilts  : ty=+0.00000°, tz=+0.00000°
p0..p4 : (0.0, 0.0, 0.0, 0.0, 0.0)


## Step 2 — Rectify

`correct_image` inverts the distortion via a 2-step fixed-point
iteration + bilinear interpolation. For panel detectors (Pilatus, Eiger),
pass a `panels=...` list and a `panel_shifts=...` file from the
calibration output.

In [3]:
import tifffile
raw = tifffile.imread(img_path).astype(np.float64)

rectified = mi.correct_image(
    img_path, result.geometry,
    nr_pixels_y=1475, nr_pixels_z=1679,
)

print(f'raw       : shape={raw.shape}, dtype={raw.dtype}, max={raw.max():.0f}')
print(f'rectified : shape={rectified.shape}, dtype={rectified.dtype}, max={rectified.max():.0f}')
print(f'max |Δ|   : {np.abs(rectified - raw).max():.0f} counts')

raw       : shape=(1679, 1475), dtype=float64, max=2340557
rectified : shape=(1679, 1475), dtype=float64, max=2340557
max |Δ|   : 10096 counts


## Step 3 — Save with provenance metadata

`write_tiff` embeds the geometry summary in the TIFF's `ImageDescription`
tag so downstream tools (and your future self) can trace which
calibration produced the rectified image.

In [4]:
out = workdir / 'CeO2_rectified.tif'
mi.write_tiff(out, rectified, geometry=result.geometry)

with tifffile.TiffFile(out) as tf:
    desc = tf.pages[0].tags['ImageDescription'].value
print('Provenance stamp:')
print(' ', desc)

Provenance stamp:
  midas-integrate v0.1.0 rectified image; Lsd=657436.900 um, BC=(685.500, 921.000) px, px=172.000 um


## Step 4 — Batch form for whole scans

In [5]:
# In practice you'd point this at a glob of scan frames.
written = mi.correct_images(
    [img_path], result.geometry,
    output_dir=workdir / 'rectified',
)
for p in written:
    print(f'  {p}')

  /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mi_correct_l01nhfdq/rectified/CeO2_00001_corrected.tif
